In [0]:
# Configure ADLS access

spark.conf.set(
    "fs.azure.account.key.@storageaccount.dfs.core.windows.net",
    "Access token"
)

In [0]:
from pyspark.sql.functions import col, avg, sum, max, min, count, round

In [0]:
# ============================================================
# READ SILVER DATASETS
# ============================================================

weather_silver_path = "abfss://processed@smartagrinil.dfs.core.windows.net/silver/weather"
soil_silver_path = "abfss://processed@smartagrinil.dfs.core.windows.net/silver/soil"
crop_silver_path = "abfss://processed@smartagrinil.dfs.core.windows.net/silver/crop_yield"
market_silver_path = "abfss://processed@smartagrinil.dfs.core.windows.net/silver/market_price"

weather = spark.read.format("delta").load(weather_silver_path)
soil = spark.read.format("delta").load(soil_silver_path)
crop = spark.read.format("delta").load(crop_silver_path)
market = spark.read.format("delta").load(market_silver_path)

print("Weather")
weather.printSchema()

print("Soil")
soil.printSchema()

print("Crop Yield")
crop.printSchema()

print("Market Price")
market.printSchema()

Weather
root
 |-- date: date (nullable = true)
 |-- district: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- temperature_c: double (nullable = true)
 |-- precipitation_mm: double (nullable = true)
 |-- relative_humidity_pct: double (nullable = true)
 |-- solar_radiation_kwh_m2: double (nullable = true)

Soil
root
 |-- sample_date: date (nullable = true)
 |-- state: string (nullable = true)
 |-- district: string (nullable = true)
 |-- soil_type: string (nullable = true)
 |-- ph: double (nullable = true)
 |-- organic_carbon_pct: double (nullable = true)
 |-- nitrogen_kg_ha: double (nullable = true)
 |-- phosphorus_kg_ha: double (nullable = true)
 |-- potassium_kg_ha: double (nullable = true)
 |-- moisture_pct: double (nullable = true)

Crop Yield
root
 |-- state: string (nullable = true)
 |-- district: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- crop: string (nullable = true)
 |-- area_harvested_ha: do

In [0]:
# ============================================================
# GOLD 1: CROP YIELD + MARKET PRICE
# ============================================================

# Aggregate market prices by state, district and commodity
market_summary = (
    market
    .groupBy(
        "state",
        "district",
        "commodity"
    )
    .agg(
        round(avg("min_price_per_quintal"), 2).alias("avg_min_price"),
        round(avg("max_price_per_quintal"), 2).alias("avg_max_price"),
        round(avg("modal_price_per_quintal"), 2).alias("avg_modal_price")
    )
)

# Join crop yield with market information
crop_market_gold = (
    crop.alias("c")
    .join(
        market_summary.alias("m"),
        (
            (col("c.state") == col("m.state")) &
            (col("c.district") == col("m.district")) &
            (col("c.crop") == col("m.commodity"))
        ),
        "left"
    )
    .select(
        col("c.state"),
        col("c.district"),
        col("c.year"),
        col("c.crop"),
        col("c.area_harvested_ha"),
        col("c.production_tonnes"),
        col("c.yield_kg_per_ha"),
        col("m.avg_min_price"),
        col("m.avg_max_price"),
        col("m.avg_modal_price")
    )
)

print("Crop + Market Gold:")
print("Rows:", crop_market_gold.count())

display(crop_market_gold)

Crop + Market Gold:
Rows: 1500


state,district,year,crop,area_harvested_ha,production_tonnes,yield_kg_per_ha,avg_min_price,avg_max_price,avg_modal_price
Maharashtra,Kolhapur,2019,Cotton,38694.37,100286.28,1929.84,2030.37,3003.06,2486.13
Maharashtra,Aurangabad,2025,Onion,25845.09,855314.08,4997.74,2008.91,2910.35,2384.75
Maharashtra,Sangli,2024,Potato,9481.7,643597.21,1624.22,1833.52,2640.12,2201.18
Maharashtra,Satara,2021,Potato,19600.73,223004.57,2112.25,2120.1,3087.16,2593.34
Maharashtra,Solapur,2022,Wheat,48810.99,1220023.79,1875.83,1760.33,2729.22,2216.24
Maharashtra,Aurangabad,2021,Soybean,2607.89,1100327.77,4617.86,1792.63,2678.03,2229.51
Maharashtra,Sangli,2021,Tomato,3977.53,316805.41,4542.14,2402.38,3319.4,2831.04
Maharashtra,Nashik,2020,Maize,8536.76,188007.15,5424.66,2229.78,3111.72,2668.94
Maharashtra,Aurangabad,2022,Onion,37759.82,124741.76,2814.6,2008.91,2910.35,2384.75
Maharashtra,Aurangabad,2018,Chickpea,20176.58,1295547.9,556.97,1270.53,2199.6,1536.2


In [0]:
crop_market_gold_path = "abfss://processed@smartagrinil.dfs.core.windows.net/gold/crop_market_summary"

crop_market_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(crop_market_gold_path)

print("Crop Market Gold created successfully!")

Crop Market Gold created successfully!


In [0]:
# ============================================================
# GOLD 2: WEATHER + SOIL
# ============================================================

# Aggregate weather information by district
weather_summary = (
    weather
    .groupBy("district")
    .agg(
        round(avg("temperature_c"), 2).alias("avg_temperature_c"),
        round(avg("precipitation_mm"), 2).alias("avg_precipitation_mm"),
        round(avg("relative_humidity_pct"), 2).alias("avg_humidity_pct"),
        round(avg("solar_radiation_kwh_m2"), 2).alias("avg_solar_radiation")
    )
)

# Aggregate soil information by state and district
soil_summary = (
    soil
    .groupBy(
        "state",
        "district"
    )
    .agg(
        round(avg("ph"), 2).alias("avg_soil_ph"),
        round(avg("organic_carbon_pct"), 2).alias("avg_organic_carbon_pct"),
        round(avg("nitrogen_kg_ha"), 2).alias("avg_nitrogen_kg_ha"),
        round(avg("phosphorus_kg_ha"), 2).alias("avg_phosphorus_kg_ha"),
        round(avg("potassium_kg_ha"), 2).alias("avg_potassium_kg_ha"),
        round(avg("moisture_pct"), 2).alias("avg_soil_moisture_pct")
    )
)

# Join Weather and Soil
weather_soil_gold = (
    soil_summary.alias("s")
    .join(
        weather_summary.alias("w"),
        col("s.district") == col("w.district"),
        "left"
    )
    .select(
        col("s.state"),
        col("s.district"),
        col("s.avg_soil_ph"),
        col("s.avg_organic_carbon_pct"),
        col("s.avg_nitrogen_kg_ha"),
        col("s.avg_phosphorus_kg_ha"),
        col("s.avg_potassium_kg_ha"),
        col("s.avg_soil_moisture_pct"),
        col("w.avg_temperature_c"),
        col("w.avg_precipitation_mm"),
        col("w.avg_humidity_pct"),
        col("w.avg_solar_radiation")
    )
)

print("Weather + Soil Gold:")
print("Rows:", weather_soil_gold.count())

display(weather_soil_gold)

Weather + Soil Gold:
Rows: 10


state,district,avg_soil_ph,avg_organic_carbon_pct,avg_nitrogen_kg_ha,avg_phosphorus_kg_ha,avg_potassium_kg_ha,avg_soil_moisture_pct,avg_temperature_c,avg_precipitation_mm,avg_humidity_pct,avg_solar_radiation
Maharashtra,Solapur,6.82,0.67,256.24,23.16,244.03,37.94,27.34,8.91,63.58,5.07
Maharashtra,Pune,6.84,0.65,265.06,22.02,254.18,35.97,27.16,9.93,61.19,5.21
Maharashtra,Aurangabad,6.89,0.65,239.25,22.51,246.14,37.54,27.05,9.26,63.39,5.33
Maharashtra,Dhule,6.75,0.68,259.56,22.44,262.25,38.73,27.22,9.36,60.61,5.33
Maharashtra,Kolhapur,6.77,0.66,261.35,21.58,257.12,37.62,26.83,8.56,60.72,5.36
Maharashtra,Nashik,6.75,0.62,268.22,22.06,268.89,38.24,27.52,9.52,61.06,5.18
Maharashtra,Sangli,6.9,0.66,270.1,21.31,262.43,38.44,27.26,10.6,60.81,5.22
Maharashtra,Jalgaon,6.83,0.67,264.0,21.67,259.62,37.49,27.01,8.95,61.33,5.26
Maharashtra,Ahmednagar,6.81,0.69,256.43,21.99,267.42,39.37,28.0,9.52,60.58,5.24
Maharashtra,Satara,6.75,0.65,263.17,21.92,261.07,37.11,26.02,9.34,63.55,5.1


In [0]:
weather_soil_gold_path = "abfss://processed@smartagrinil.dfs.core.windows.net/gold/agriculture_weather_soil_summary"

weather_soil_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(weather_soil_gold_path)

print("Weather Soil Gold created successfully!")

Weather Soil Gold created successfully!


In [0]:
# ============================================================
# GOLD LAYER VERIFICATION
# ============================================================

print("========================================")
print("GOLD LAYER VERIFICATION")
print("========================================")

print("Crop Market rows:",
      spark.read.format("delta")
      .load(crop_market_gold_path)
      .count())

print("Weather Soil rows:",
      spark.read.format("delta")
      .load(weather_soil_gold_path)
      .count())

print("========================================")
print("GOLD LAYER COMPLETED!")
print("========================================")

GOLD LAYER VERIFICATION
Crop Market rows: 1500
Weather Soil rows: 10
GOLD LAYER COMPLETED!


In [0]:
display(
    spark.read
    .format("delta")
    .load(crop_market_gold_path)
)

state,district,year,crop,area_harvested_ha,production_tonnes,yield_kg_per_ha,avg_min_price,avg_max_price,avg_modal_price
Maharashtra,Kolhapur,2019,Cotton,38694.37,100286.28,1929.84,2030.37,3003.06,2486.13
Maharashtra,Aurangabad,2025,Onion,25845.09,855314.08,4997.74,2008.91,2910.35,2384.75
Maharashtra,Sangli,2024,Potato,9481.7,643597.21,1624.22,1833.52,2640.12,2201.18
Maharashtra,Satara,2021,Potato,19600.73,223004.57,2112.25,2120.1,3087.16,2593.34
Maharashtra,Solapur,2022,Wheat,48810.99,1220023.79,1875.83,1760.33,2729.22,2216.24
Maharashtra,Aurangabad,2021,Soybean,2607.89,1100327.77,4617.86,1792.63,2678.03,2229.51
Maharashtra,Sangli,2021,Tomato,3977.53,316805.41,4542.14,2402.38,3319.4,2831.04
Maharashtra,Nashik,2020,Maize,8536.76,188007.15,5424.66,2229.78,3111.72,2668.94
Maharashtra,Aurangabad,2022,Onion,37759.82,124741.76,2814.6,2008.91,2910.35,2384.75
Maharashtra,Aurangabad,2018,Chickpea,20176.58,1295547.9,556.97,1270.53,2199.6,1536.2


In [0]:
display(
    spark.read
    .format("delta")
    .load(weather_soil_gold_path)
)

state,district,avg_soil_ph,avg_organic_carbon_pct,avg_nitrogen_kg_ha,avg_phosphorus_kg_ha,avg_potassium_kg_ha,avg_soil_moisture_pct,avg_temperature_c,avg_precipitation_mm,avg_humidity_pct,avg_solar_radiation
Maharashtra,Solapur,6.82,0.67,256.24,23.16,244.03,37.94,27.34,8.91,63.58,5.07
Maharashtra,Pune,6.84,0.65,265.06,22.02,254.18,35.97,27.16,9.93,61.19,5.21
Maharashtra,Aurangabad,6.89,0.65,239.25,22.51,246.14,37.54,27.05,9.26,63.39,5.33
Maharashtra,Dhule,6.75,0.68,259.56,22.44,262.25,38.73,27.22,9.36,60.61,5.33
Maharashtra,Kolhapur,6.77,0.66,261.35,21.58,257.12,37.62,26.83,8.56,60.72,5.36
Maharashtra,Nashik,6.75,0.62,268.22,22.06,268.89,38.24,27.52,9.52,61.06,5.18
Maharashtra,Sangli,6.9,0.66,270.1,21.31,262.43,38.44,27.26,10.6,60.81,5.22
Maharashtra,Jalgaon,6.83,0.67,264.0,21.67,259.62,37.49,27.01,8.95,61.33,5.26
Maharashtra,Ahmednagar,6.81,0.69,256.43,21.99,267.42,39.37,28.0,9.52,60.58,5.24
Maharashtra,Satara,6.75,0.65,263.17,21.92,261.07,37.11,26.02,9.34,63.55,5.1
